# Implied Volatility from Bid, Mid, and Ask Prices

Implied volatility is the volatility input that makes the Black-Scholes model match an observed market option price. This notebook solves implied volatility from bid, mid, and ask prices so that quote uncertainty is preserved instead of collapsed into one number.

## Forward and inverse problems

The forward Black-Scholes problem is:

\[
S, K, T, r, \sigma \rightarrow \text{option price}
\]

The implied-volatility problem reverses that relationship:

\[
S, K, T, r, \text{option price} \rightarrow \sigma
\]

There is no simple closed-form formula for implied volatility, so the value is solved numerically.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.implied_vol import (
    build_iv_wide_table,
    calculate_iv_results,
    calculate_iv_uncertainty,
    plot_solver_failure_heatmap,
    solver_failure_summary,
    solver_success_summary,
    vendor_iv_comparison,
)

## Load cleaned option quotes

The input is the cleaned quote file from the quote-quality step. Excluded quotes remain in the file for auditability, but the solver uses retained quotes by default.

In [ ]:
cleaned_path = project_root / "data" / "processed" / "cleaned_option_chain.csv"

if not cleaned_path.exists():
    raise FileNotFoundError("Cleaned option-chain file not found.")

cleaned_options = pd.read_csv(cleaned_path)
cleaned_options.head(10)

## Solve IV from bid, mid, and ask

The risk-free rate is treated as an input assumption. For a full empirical run, this value should be aligned with the snapshot date and maturity range.

In [ ]:
risk_free_rate = 0.04

iv_results = calculate_iv_results(
    cleaned_options,
    risk_free_rate=risk_free_rate,
    method="bisection",
    lower_bound=1e-6,
    upper_bound=5.0,
    tolerance=1e-8,
    max_iterations=100,
    retained_only=True,
)

iv_results.head(10)

In [ ]:
iv_results_path = project_root / "data" / "processed" / "iv_results.csv"
iv_results.to_csv(iv_results_path, index=False)

iv_results_path

## Solver success rate

In [ ]:
success_summary = solver_success_summary(iv_results)
success_summary

## Solver failure summary

Solver failures are not only coding issues. They can also indicate stale quotes, prices outside no-arbitrage bounds, deep out-of-the-money contracts with weak prices, or quotes whose bid and ask range is too noisy to support a stable IV estimate.

In [ ]:
failure_summary = solver_failure_summary(iv_results)
failure_summary

## IV bid, mid, and ask columns

The wide table gives one row per option contract with separate IV values from bid, mid, and ask prices.

In [ ]:
iv_wide = build_iv_wide_table(cleaned_options, iv_results)
iv_wide = calculate_iv_uncertainty(iv_wide)

iv_columns = [
    column
    for column in [
        "contractSymbol",
        "expiration",
        "option_type",
        "strike",
        "bid",
        "mid_price",
        "ask",
        "IV_bid",
        "IV_mid",
        "IV_ask",
        "IV_range",
        "IV_relative_range",
    ]
    if column in iv_wide.columns
]

iv_wide[iv_columns].head(15)

In [ ]:
iv_uncertainty_path = project_root / "data" / "processed" / "iv_uncertainty_results.csv"
iv_wide.to_csv(iv_uncertainty_path, index=False)

iv_uncertainty_path

## Computed IV versus vendor IV

The vendor IV field is useful as a reference, but the project computes its own bid, mid, and ask implied volatilities from observed quote prices.

In [ ]:
vendor_comparison = vendor_iv_comparison(iv_wide)
vendor_comparison.head(15)

In [ ]:
vendor_comparison_path = project_root / "data" / "processed" / "computed_vs_vendor_iv.csv"

if not vendor_comparison.empty:
    vendor_comparison.to_csv(vendor_comparison_path, index=False)

vendor_comparison_path

## Solver failure heatmap

The heatmap groups solver failure rates by expiry and log-moneyness. This helps identify where implied-volatility estimates are least reliable.

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

heatmap_path = plot_solver_failure_heatmap(
    cleaned_options,
    iv_results,
    figures_dir / "solver_failure_heatmap.png",
    moneyness_bins=8,
)

heatmap_path

## Database-ready outputs

The long IV result table can be inserted into the `iv_results` table. The wide uncertainty table can be used to populate `iv_uncertainty_results` after joining to cleaned quote identifiers.